**Objetivo:** Preparar X, y e os folds de validação cruzada (`StratifiedGroupKFold`, grupos = `SITE_ID`).

**Entradas:** `Data/conectivity/connectivity_matrices_abide_validado.pkl` (notebook 05), `Data/raw/phenotypic_abide.csv`.

**Processamento:** Vetorização, `y_bin = (y == 1)` (0 = Controle, 1 = TEA), 5 folds com `random_state=42`. O `StandardScaler` NÃO é ajustado aqui (evita data leakage; é ajustado dentro de cada fold).

**Saídas:** `Data/processed/dataset_abide_processado.pkl`.

In [ ]:
# ------------------------------------------------------------
# Configuração do projeto (execução local — VS Code)
# Garante que a raiz do projeto esteja no sys.path, funcionando tanto se o
# diretório de trabalho do kernel for notebooks/ quanto a raiz do projeto.
# ------------------------------------------------------------
import sys
from pathlib import Path

_raiz = Path.cwd().resolve()
for _cand in (_raiz, *_raiz.parents):
    if (_cand / 'src' / 'config.py').exists():
        _raiz = _cand
        break
else:
    raise RuntimeError(
        'Não foi possível localizar a pasta src/config.py a partir de '
        f'{Path.cwd()}. Abra a pasta TCC_TEA_TDAH/ no VS Code.'
    )
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

import os
import random
import numpy as np

from src.config import BASE_DIR, PATHS, SEED, criar_pastas
from src.utils import exigir_arquivo

criar_pastas()

random.seed(SEED)
np.random.seed(SEED)

print('Projeto (BASE_DIR) =', BASE_DIR)

In [2]:
exigir_arquivo(PATHS['conectivity'] / 'connectivity_matrices_abide_validado.pkl', '05_Validacao_ABIDE.ipynb')
exigir_arquivo(PATHS['raw'] / 'phenotypic_abide.csv', '02_Download_ABIDE.ipynb')

import pickle
import numpy as np
import pandas as pd
from nilearn.connectome import sym_matrix_to_vec
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

with open(os.path.join(PATHS['conectivity'], 'connectivity_matrices_abide_validado.pkl'), 'rb') as f:
    conn_dict = pickle.load(f)

phenotypic = pd.read_csv(os.path.join(PATHS['raw'], 'phenotypic_abide.csv')).set_index('SUB_ID')

sub_ids = np.array(list(conn_dict.keys()))
matrices = np.array([conn_dict[s] for s in sub_ids])

X = sym_matrix_to_vec(matrices, discard_diagonal=True)          # features vetorizadas
y = phenotypic.loc[sub_ids, 'DX_GROUP'].values                  # 1=TEA, 2=Controle
groups = phenotypic.loc[sub_ids, 'SITE_ID'].values               # site de aquisição

print('X:', X.shape, '| y:', y.shape, '| sites únicos:', len(np.unique(groups)))

X: (871, 19900) | y: (871,) | sites únicos: 20


In [3]:
# Recodifica os rótulos para 0/1 (convenção mais comum em scikit-learn/XGBoost)
# 0 = Controle, 1 = TEA

y_bin = (y == 1).astype(int)
print('Distribuição y_bin:', np.bincount(y_bin))

Distribuição y_bin: [468 403]


In [4]:
# Define os folds de validação cruzada — estratificados por classe,
# agrupados por site. n_splits=5 é um valor padrão que equilibra
# robustez da estimativa com tamanho suficiente de amostra de teste por fold.
N_SPLITS = 5
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

folds = list(sgkf.split(X, y_bin, groups))

for i, (train_idx, test_idx) in enumerate(folds):
    sites_treino = set(groups[train_idx])
    sites_teste = set(groups[test_idx])
    intersecao = sites_treino & sites_teste
    print(f'Fold {i}: treino={len(train_idx)} teste={len(test_idx)} | '
          f'sites em comum entre treino/teste: {len(intersecao)} (deve ser 0)')

Fold 0: treino=830 teste=41 | sites em comum entre treino/teste: 0 (deve ser 0)
Fold 1: treino=698 teste=173 | sites em comum entre treino/teste: 0 (deve ser 0)
Fold 2: treino=706 teste=165 | sites em comum entre treino/teste: 0 (deve ser 0)
Fold 3: treino=547 teste=324 | sites em comum entre treino/teste: 0 (deve ser 0)
Fold 4: treino=703 teste=168 | sites em comum entre treino/teste: 0 (deve ser 0)


In [5]:
# --------------------------------------------------------
# Salva o dataset processado + definição dos folds — pronto para os
# notebooks 08 (baselines) e 09 (deep learning) consumirem diretamente
# --------------------------------------------------------
dataset_processado = {
    'X': X,
    'y': y_bin,
    'groups': groups,
    'sub_ids': sub_ids,
    'folds': folds,          # lista de (train_idx, test_idx)
    'n_splits': N_SPLITS,
    'seed': SEED,
}

out_path = os.path.join(PATHS['processed'], 'dataset_abide_processado.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(dataset_processado, f)

print('Dataset processado salvo em:', out_path)
print('OBS: o StandardScaler NÃO é salvo aqui — ele deve ser ajustado dentro de')
print('cada fold, no notebook 08, para evitar data leakage.')

Dataset processado salvo em: <BASE_DIR>/Data/processed/dataset_abide_processado.pkl
OBS: o StandardScaler NÃO é salvo aqui — ele deve ser ajustado dentro de
cada fold, no notebook 08, para evitar data leakage.
